# Pandas ile Veri Temizleme (Grocery Chain Verisi ile)

Bu notebook, `grocery_chain_data.json` dosyasındaki **gerçek veri kalitesi sorunlarını** kullanarak pandas ile veri temizleme sürecini adım adım gösterir.

Veri setinde tespit edilen gerçek sorunlar:
- **Eksik mağaza adları** (`store_name` sütununda boş değerler)
- **Mantıksız/negatif tutarlar** (`discount_amount`, `total_amount`'tan büyük olduğunda `final_amount` negatif çıkıyor)
- **Metin olarak saklanan tarih sütunu** (`transaction_date` datetime değil, string)

**İçerik:**
1. Veriyi Yükleme ve İlk Bakış
2. Genel Veri Sağlığı Kontrolü
3. Eksik Değerlerin Tespiti ve Giderilmesi
4. Yinelenen (Duplicate) Kayıt Kontrolü
5. Veri Tiplerinin Düzeltilmesi (Tarih Sütunu)
6. Mantıksal Hata / Aykırı Değer Tespiti ve Düzeltilmesi
7. Kategori Tutarlılığı Kontrolü
8. İndeks Düzenleme
9. Temizlenmiş Veriyi Kaydetme
10. Uçtan Uca Temizleme Fonksiyonu

> **Not:** Bu notebook'u çalıştırmak için `grocery_chain_data.json` dosyasının notebook ile **aynı klasörde** olması gerekir.

## 1. Veriyi Yükleme ve İlk Bakış

JSON dosyasını okuyoruz. JSON formatı Excel'e göre veri tiplerini daha az koruduğu için (örn. tarihler string olarak kalır), temizlik açısından daha öğretici bir başlangıç noktasıdır.

In [1]:
import pandas as pd
import numpy as np

df_ham = pd.read_json('grocery_chain_data.json')
print("Boyut:", df_ham.shape)
df_ham.head()

Boyut: (1980, 11)


,customer_id,store_name,transaction_date,aisle,product_name,quantity,unit_price,total_amount,discount_amount,final_amount,loyalty_points
0,2824,GreenGrocer Plaza,2023-08-26T00:00:00.000,Produce,Pasta,2.0,7.46,14.92,0.00,14.92,377
1,5506,ValuePlus Market,2024-02-13T00:00:00.000,Dairy,Cheese,1.0,1.85,1.85,3.41,-1.56,111
2,4657,ValuePlus Market,2023-11-23T00:00:00.000,Bakery,Onions,4.0,7.38,29.52,4.04,25.48,301
3,2679,SuperSave Central,2025-01-13T00:00:00.000,Snacks & Candy,Cereal,3.0,5.50,16.50,1.37,15.13,490
4,9935,GreenGrocer Plaza,2023-10-13T00:00:00.000,Canned Goods,Orange Juice,5.0,8.66,43.30,1.50,41.80,22


## 2. Genel Veri Sağlığı Kontrolü

Temizliğe başlamadan önce verinin genel durumunu anlamak gerekir:

- `df.info()` → veri tipleri ve boş değer sayıları
- `df.isnull().sum()` → sütun bazında eksik değer sayısı
- `df.duplicated().sum()` → tekrar eden satır sayısı

In [2]:
df_ham.info()

<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       1980 non-null   int64  
 1   store_name        1955 non-null   str    
 2   transaction_date  1980 non-null   str    
 3   aisle             1980 non-null   str    
 4   product_name      1980 non-null   str    
 5   quantity          1980 non-null   float64
 6   unit_price        1980 non-null   float64
 7   total_amount      1980 non-null   float64
 8   discount_amount   1980 non-null   float64
 9   final_amount      1980 non-null   float64
 10  loyalty_points    1980 non-null   int64  
dtypes: float64(5), int64(2), str(4)
memory usage: 170.3 KB


In [3]:
print("Sütun bazında eksik değer sayısı:")
print(df_ham.isnull().sum())

print("\nToplam tekrar eden satır sayısı:", df_ham.duplicated().sum())

Sütun bazında eksik değer sayısı:
customer_id          0
store_name          25
transaction_date     0
aisle                0
product_name         0
quantity             0
unit_price           0
total_amount         0
discount_amount      0
final_amount         0
loyalty_points       0
dtype: int64

Toplam tekrar eden satır sayısı: 0


## 3. Eksik Değerlerin Tespiti ve Giderilmesi

`store_name` sütununda eksik değerler olduğunu gördük. Bu satırları incelediğimizde, diğer tüm sütunların (ürün, tutar, sadakat puanı vb.) dolu olduğunu görüyoruz — yani sadece mağaza bilgisi kaybolmuş.

Burada iki mantıklı seçenek var:
1. **Satırları silmek** (eğer hangi mağazada olduğu analiz için kritikse ve tahmin edilemiyorsa)
2. **`"Bilinmiyor"` gibi bir yer tutucu ile doldurmak** (eğer diğer sütunlar hâlâ analiz için değerliyse, örn. ürün bazlı analiz yapılacaksa satırı tamamen kaybetmek istemeyiz)

Biz burada veriyi kaybetmemek için ikinci yolu seçiyoruz ve bu kararı **açıkça belirtiyoruz**.

In [4]:
# Mağaza adı eksik olan satırları inceleyelim
eksik_magaza = df_ham[df_ham['store_name'].isnull()]
print(f"Mağaza adı eksik olan satır sayısı: {len(eksik_magaza)}")
eksik_magaza.head()

Mağaza adı eksik olan satır sayısı: 25


,customer_id,store_name,transaction_date,aisle,product_name,quantity,unit_price,total_amount,discount_amount,final_amount,loyalty_points
285,9445,NaN,2025-01-03T00:00:00.000,Canned Goods,Bread,5.0,29.82,149.10,14.91,134.19,478
385,2983,NaN,2025-04-16T00:00:00.000,Frozen Foods,Milk,2.0,15.34,30.68,3.07,27.61,57
389,9289,NaN,2024-12-21T00:00:00.000,Snacks & Candy,Cheese,4.0,3.38,13.52,2.03,11.49,274
418,7658,NaN,2024-08-20T00:00:00.000,Dairy,Milk,4.0,2.46,9.84,0.00,9.84,473
540,1472,NaN,2024-12-08T00:00:00.000,Household Items,Milk,5.0,25.44,127.20,3.20,124.00,154


In [5]:
df_temiz = df_ham.copy()

# Eksik mağaza adlarını "Bilinmiyor" ile dolduruyoruz (satırı silmek yerine)
# Not: Bu karar, ürün/reyon bazlı analizlerde bu satırların hâlâ kullanılabilir olmasını sağlar
df_temiz['store_name'] = df_temiz['store_name'].fillna('Bilinmiyor')

print("Doldurma sonrası eksik değer sayısı:", df_temiz['store_name'].isnull().sum())
df_temiz['store_name'].value_counts()

Doldurma sonrası eksik değer sayısı: 0


store_name
City Fresh Store      235
SuperSave Central     232
ValuePlus Market      221
GreenGrocer Plaza     220
Corner Grocery        218
FamilyFood Express    215
MegaMart Westside     214
QuickStop Market      208
FreshMart Downtown    192
Bilinmiyor             25
Name: count, dtype: int64

## 4. Yinelenen (Duplicate) Kayıt Kontrolü

Yukarıda `df.duplicated().sum()` ile tam satır bazında tekrar kontrolü yaptık ve tekrar eden kayıt bulunmadığını gördük. Yine de iyi bir pratik olarak, **anlamlı bir alt küme** üzerinden de kontrol etmek faydalıdır: örneğin aynı müşteri, aynı tarih ve aynı ürünle birden fazla kayıt varsa bu bir veri girişi hatası olabilir.

In [6]:
# Anlamlı bir alt kümeye göre (müşteri + tarih + ürün) tekrar kontrolü
tekrar_sayisi = df_temiz.duplicated(subset=['customer_id', 'transaction_date', 'product_name']).sum()
print(f"Müşteri+tarih+ürün bazında tekrar eden kayıt sayısı: {tekrar_sayisi}")

# Bu veri setinde tekrar bulunmuyor, ancak bulunsaydı şu şekilde silinirdi:
# df_temiz = df_temiz.drop_duplicates(subset=['customer_id', 'transaction_date', 'product_name'], keep='first')

Müşteri+tarih+ürün bazında tekrar eden kayıt sayısı: 0


## 5. Veri Tiplerinin Düzeltilmesi (Tarih Sütunu)

`df.info()` çıktısında `transaction_date` sütununun `str` (metin) tipinde olduğunu görmüştük. Tarih üzerinde filtreleme, sıralama veya "hangi ay en çok satış oldu" gibi analizler yapabilmek için bu sütunu gerçek bir **datetime** tipine çevirmemiz gerekir.

- `pd.to_datetime()` → metni datetime tipine çevirir.
- `errors='coerce'` → çevrilemeyen değerleri hataya düşmek yerine `NaT` (Not a Time) yapar, böylece kod durmaz ve sorunlu satırlar sonradan tespit edilebilir.

In [7]:
print("Dönüşümden önce tip:", df_temiz['transaction_date'].dtype)
print("Örnek değer:", df_temiz['transaction_date'].iloc[0])

df_temiz['transaction_date'] = pd.to_datetime(df_temiz['transaction_date'], errors='coerce')

print("\nDönüşümden sonra tip:", df_temiz['transaction_date'].dtype)
print("Çevrilemeyen (NaT) satır sayısı:", df_temiz['transaction_date'].isnull().sum())

# Artık tarihten yıl/ay gibi bilgiler çıkarabiliriz
df_temiz['islem_yili'] = df_temiz['transaction_date'].dt.year
df_temiz['islem_ayi'] = df_temiz['transaction_date'].dt.month
df_temiz[['transaction_date', 'islem_yili', 'islem_ayi']].head()

Dönüşümden önce tip: str
Örnek değer: 2023-08-26T00:00:00.000

Dönüşümden sonra tip: datetime64[us]
Çevrilemeyen (NaT) satır sayısı: 0


,transaction_date,islem_yili,islem_ayi
0,2023-08-26,2023,8
1,2024-02-13,2024,2
2,2023-11-23,2023,11
3,2025-01-13,2025,1
4,2023-10-13,2023,10


## 6. Mantıksal Hata / Aykırı Değer Tespiti ve Düzeltilmesi

Veride şöyle bir iş kuralı olması beklenir: `final_amount = total_amount - discount_amount` ve bu değer **negatif olamaz** (bir alışverişten müşteriye para iadesi anlamına gelmez, bu senaryoda). Ancak bazı satırlarda `discount_amount`, `total_amount`'tan büyük olduğu için `final_amount` negatif çıkmış — bu bir **veri/iş mantığı hatasıdır**.

Çözüm yaklaşımı:
1. Önce bu satırları tespit edip incele.
2. İndirim tutarını, toplam tutarı geçemeyecek şekilde sınırla (`discount_amount = min(discount_amount, total_amount)`).
3. `final_amount`'ı yeniden hesapla.

Bu, veriyi olduğu gibi silmek yerine **iş kuralına göre düzeltmeyi** tercih ettiğimiz bir örnektir; çünkü işlemin kendisi (müşteri, ürün, adet) geçerli, sadece indirim hesaplaması hatalı.

In [8]:
# Mantıksız satırları tespit etme: indirim, toplam tutardan büyük mü?
mantiksiz = df_temiz[df_temiz['discount_amount'] > df_temiz['total_amount']]
print(f"Mantıksız (negatif final_amount üreten) satır sayısı: {len(mantiksiz)}")
mantiksiz[['product_name', 'total_amount', 'discount_amount', 'final_amount']]

Mantıksız (negatif final_amount üreten) satır sayısı: 13


,product_name,total_amount,discount_amount,final_amount
1,Cheese,1.85,3.41,-1.56
28,Onions,2.46,4.71,-2.25
60,Yogurt,3.40,4.40,-1.00
665,Cereal,3.78,4.34,-0.56
773,Onions,1.01,4.44,-3.43
831,Carrots,3.04,3.32,-0.28
856,Yogurt,1.63,3.58,-1.95
1080,Yogurt,3.97,4.16,-0.19
1291,Bananas,3.97,4.55,-0.58
1362,Bread,1.73,3.47,-1.74


In [9]:
# Düzeltme: indirim tutarını toplam tutarla sınırla, final_amount'ı yeniden hesapla
df_temiz['discount_amount'] = np.minimum(df_temiz['discount_amount'], df_temiz['total_amount'])
df_temiz['final_amount'] = df_temiz['total_amount'] - df_temiz['discount_amount']

# Kontrol: negatif final_amount kaldı mı?
print("Düzeltme sonrası negatif final_amount sayısı:", (df_temiz['final_amount'] < 0).sum())

Düzeltme sonrası negatif final_amount sayısı: 0


## 7. Kategori Tutarlılığı Kontrolü

Kategorik sütunlarda (mağaza adı, reyon, ürün adı) yazım tutarsızlığı olup olmadığını kontrol etmek iyi bir alışkanlıktır — örneğin `"istanbul"` ve `"İSTANBUL"` gibi aynı kategoriyi farklı gösteren değerler.

`.unique()` ile tüm benzersiz değerleri, `.str.strip()` ile de baştaki/sondaki boşluk sorunlarını kontrol edebiliriz.

In [10]:
# Reyon (aisle) ve mağaza adlarında tutarsızlık var mı kontrol edelim
print("Reyonlar:", sorted(df_temiz['aisle'].unique()))
print("\nMağazalar:", sorted(df_temiz['store_name'].unique()))

# Baştaki/sondaki boşluk kontrolü
bosluk_sorunu = df_temiz['store_name'].apply(lambda x: x != x.strip()).sum()
print(f"\nBaşta/sonda boşluk içeren mağaza adı sayısı: {bosluk_sorunu}")

Reyonlar: ['Bakery', 'Beverages', 'Canned Goods', 'Dairy', 'Frozen Foods', 'Health & Wellness', 'Household Items', 'Meat & Seafood', 'Personal Care', 'Produce', 'Snacks & Candy']

Mağazalar: ['Bilinmiyor', 'City Fresh Store', 'Corner Grocery', 'FamilyFood Express', 'FreshMart Downtown', 'GreenGrocer Plaza', 'MegaMart Westside', 'QuickStop Market', 'SuperSave Central', 'ValuePlus Market']

Başta/sonda boşluk içeren mağaza adı sayısı: 0


In [11]:
# Kontrol sonucu: bu veri setinde kategori tutarsızlığı bulunmuyor (hepsi standart yazılmış).
# Yine de, bulunsaydı şu şekilde düzeltilirdi:
# df_temiz['store_name'] = df_temiz['store_name'].str.strip().str.title()
print("Kategori kontrolü tamamlandı: tutarsızlık bulunmadı.")

Kategori kontrolü tamamlandı: tutarsızlık bulunmadı.


## 8. İndeks Düzenleme

Bu notebook'ta satır silme işlemi yapmadık (mantıksız değerleri düzelttik, silmedik), ancak genel iyi bir pratik olarak, herhangi bir filtreleme/silme işleminden sonra indeksi sıfırdan yeniden numaralandırmak gerekir. Burada göstermek amacıyla uyguluyoruz.

In [12]:
df_temiz = df_temiz.reset_index(drop=True)
print("İndeks sıfırlandı. İlk 5 satır:")
df_temiz.head()

İndeks sıfırlandı. İlk 5 satır:


,customer_id,store_name,transaction_date,aisle,product_name,quantity,unit_price,total_amount,discount_amount,final_amount,loyalty_points,islem_yili,islem_ayi
0,2824,GreenGrocer Plaza,2023-08-26,Produce,Pasta,2.0,7.46,14.92,0.00,14.92,377,2023,8
1,5506,ValuePlus Market,2024-02-13,Dairy,Cheese,1.0,1.85,1.85,1.85,0.00,111,2024,2
2,4657,ValuePlus Market,2023-11-23,Bakery,Onions,4.0,7.38,29.52,4.04,25.48,301,2023,11
3,2679,SuperSave Central,2025-01-13,Snacks & Candy,Cereal,3.0,5.50,16.50,1.37,15.13,490,2025,1
4,9935,GreenGrocer Plaza,2023-10-13,Canned Goods,Orange Juice,5.0,8.66,43.30,1.50,41.80,22,2023,10


## 9. Temizlenmiş Veriyi Kaydetme

Temizleme işlemi bittikten sonra, sonucu ham veriden ayrı bir dosyaya kaydediyoruz. Ham JSON dosyası hiçbir zaman değiştirilmedi (tekrar üretilebilirlik için önemli).

In [13]:
df_temiz.to_csv('grocery_chain_data_temiz.csv', index=False)
print("Temizlenmiş veri 'grocery_chain_data_temiz.csv' olarak kaydedildi.")
print("\nSon durumun genel bilgisi:")
df_temiz.info()

Temizlenmiş veri 'grocery_chain_data_temiz.csv' olarak kaydedildi.

Son durumun genel bilgisi:
<class 'pandas.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       1980 non-null   int64         
 1   store_name        1980 non-null   str           
 2   transaction_date  1980 non-null   datetime64[us]
 3   aisle             1980 non-null   str           
 4   product_name      1980 non-null   str           
 5   quantity          1980 non-null   float64       
 6   unit_price        1980 non-null   float64       
 7   total_amount      1980 non-null   float64       
 8   discount_amount   1980 non-null   float64       
 9   final_amount      1980 non-null   float64       
 10  loyalty_points    1980 non-null   int64         
 11  islem_yili        1980 non-null   int32         
 12  islem_ayi         1980 non-null   int32         

## 10. Uçtan Uca Temizleme Fonksiyonu

Aynı temizleme adımlarını (örn. veri her gün/hafta yenilendiğinde) tekrar tekrar uygulamak gerekebilir. Bu yüzden tüm adımları tek bir fonksiyonda topluyoruz.

In [14]:
def veriyi_temizle(df_ham):
    """
    Ham grocery chain DataFrame'ini alır, temel temizleme adımlarını uygular
    ve temizlenmiş DataFrame'i döndürür.
    """
    df = df_ham.copy()

    # 1) Eksik mağaza adlarını doldur
    df['store_name'] = df['store_name'].fillna('Bilinmiyor')

    # 2) Tarih sütununu datetime'a çevir
    df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

    # 3) Mantıksız indirim/tutar hatasını düzelt
    df['discount_amount'] = np.minimum(df['discount_amount'], df['total_amount'])
    df['final_amount'] = df['total_amount'] - df['discount_amount']

    # 4) Tekrar eden kayıtları temizle (varsa)
    df = df.drop_duplicates(subset=['customer_id', 'transaction_date', 'product_name'], keep='first')

    # 5) İndeksi sıfırla
    df = df.reset_index(drop=True)

    return df

# Fonksiyonu ham veri üzerinde test edelim
df_sonuc = veriyi_temizle(df_ham)

print("Temizleme sonrası kontrol:")
print("- Eksik store_name:", df_sonuc['store_name'].isnull().sum())
print("- Negatif final_amount:", (df_sonuc['final_amount'] < 0).sum())
print("- transaction_date tipi:", df_sonuc['transaction_date'].dtype)
df_sonuc.head()

Temizleme sonrası kontrol:
- Eksik store_name: 0
- Negatif final_amount: 0
- transaction_date tipi: datetime64[us]


,customer_id,store_name,transaction_date,aisle,product_name,quantity,unit_price,total_amount,discount_amount,final_amount,loyalty_points
0,2824,GreenGrocer Plaza,2023-08-26,Produce,Pasta,2.0,7.46,14.92,0.00,14.92,377
1,5506,ValuePlus Market,2024-02-13,Dairy,Cheese,1.0,1.85,1.85,1.85,0.00,111
2,4657,ValuePlus Market,2023-11-23,Bakery,Onions,4.0,7.38,29.52,4.04,25.48,301
3,2679,SuperSave Central,2025-01-13,Snacks & Candy,Cereal,3.0,5.50,16.50,1.37,15.13,490
4,9935,GreenGrocer Plaza,2023-10-13,Canned Goods,Orange Juice,5.0,8.66,43.30,1.50,41.80,22


## Özet

Bu notebook'ta gerçek market zinciri verisindeki somut sorunları temizledik:

| Sorun | Tespit | Çözüm |
|---|---|---|
| Eksik mağaza adı | `.isnull().sum()` | `.fillna('Bilinmiyor')` |
| Tekrar eden kayıt riski | `.duplicated(subset=[...])` | `.drop_duplicates()` |
| Tarih string olarak saklı | `.info()`, `.dtype` | `pd.to_datetime(errors='coerce')` |
| İndirim > toplam tutar (mantık hatası) | `discount_amount > total_amount` filtresi | `np.minimum()` ile sınırlayıp yeniden hesaplama |
| Kategori tutarsızlığı | `.unique()`, `.str.strip()` kontrolü | Bu veri setinde gerek kalmadı, ama kontrol edildi |
| İndeks karışıklığı | — | `.reset_index(drop=True)` |

**Altın kural:** Ham veriyi asla doğrudan değiştirme; her zaman `.copy()` ile kopya üzerinde çalış, düzeltme kararlarını kod içinde açıkça belirt ve temizlenmiş veriyi ayrı bir dosyaya kaydet.